# Прогнозування покупки товару

In [14]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression

data = {
    "Client": ["A", "B", "C", "D", "E"],
    "Age": [25, 40, 32, 22, 45],
    "Visit": [4, 1, 7, 2, 5],
    "Mean check": [1000, 300, 1500, 200, 2000],
    "Buy Smth": [1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)

df

,Client,Age,Visit,Mean check,Buy Smth
0,A,25,4,1000,1
1,B,40,1,300,0
2,C,32,7,1500,1
3,D,22,2,200,0
4,E,45,5,2000,1


In [15]:
features = ["Age", "Visit", "Mean check"]

scaler = MinMaxScaler()

df_scaled = df.copy()
df_scaled[features] = scaler.fit_transform(df[features])

df_scaled

,Client,Age,Visit,Mean check,Buy Smth
0,A,0.130435,0.500000,0.444444,1
1,B,0.782609,0.000000,0.055556,0
2,C,0.434783,1.000000,0.722222,1
3,D,0.000000,0.166667,0.000000,0
4,E,1.000000,0.666667,1.000000,1


In [16]:
X = df_scaled[features]
y = df["Buy Smth"]

model = LogisticRegression()
model.fit(X, y)

df_scaled["Predicted Buy"] = model.predict(X)

df_scaled[["Client", "Buy Smth", "Predicted Buy"]]

,Client,Buy Smth,Predicted Buy
0,A,1,1
1,B,0,0
2,C,1,1
3,D,0,0
4,E,1,1


In [17]:
import numpy as np


w1 = 0.2
w2 = 0.3
w3 = 0.5
bias = -0.4


X_new = df_scaled[["Age", "Visit", "Mean check"]]


z = (
    w1 * X_new["Age"] +
    w2 * X_new["Visit"] +
    w3 * X_new["Mean check"] +
    bias
)


probability = 1 / (1 + np.exp(-z))


prediction = (probability >= 0.5).astype(int)


new_results = pd.DataFrame({
    "Client": df["Client"],
    "Z": z,
    "Probability": probability,
    "Predicted Buy": prediction,
    "Actual Buy": df["Buy Smth"]
})

new_results

,Client,Z,Probability,Predicted Buy,Actual Buy
0,A,-0.001691,0.499577,0,1
1,B,-0.215700,0.446283,0,0
2,C,0.348068,0.586149,1,1
3,D,-0.350000,0.413382,0,0
4,E,0.500000,0.622459,1,1


In [18]:

X = X_new.to_numpy()
y = df["Buy Smth"].to_numpy()


weights = np.array([w1, w2, w3])
bias = bias


learning_rate = 0.1
epochs = 1000


for epoch in range(epochs):

    
    z = np.dot(X, weights) + bias
    predictions = 1 / (1 + np.exp(-z))


    errors = predictions - y


    dw = np.dot(X.T, errors) / len(y)
    db = np.mean(errors)

   
    weights -= learning_rate * dw
    bias -= learning_rate * db


print("Оновлені ваги:", weights)
print("Оновлене зміщення:", bias)

Оновлені ваги: [-0.56948668  3.85620013  4.50056468]
Оновлене зміщення: -2.275392432124392


In [19]:

final_z = np.dot(X, weights) + bias

final_probability = 1 / (1 + np.exp(-final_z))

final_prediction = (final_probability >= 0.5).astype(int)


final_results = pd.DataFrame({
    "Client": df["Client"],
    "Actual Buy": y,
    "New Probability": final_probability,
    "Final Prediction": final_prediction
})

final_results

,Client,Actual Buy,New Probability,Final Prediction
0,A,1,0.829017,1
1,B,0,0.077913,0
2,C,1,0.989885,1
3,D,0,0.163462,0
4,E,1,0.985607,1
